In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os

In [64]:
folder = r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara'
# mismos agebs + 7 zonas externas
agebs_and_external_zones = gpd.read_file(os.path.join(folder, "Insumos IMEPLAN", "zonificación", "zonas ZMG_with_external_zones","zonas ZMG_with_external_Zones.shp"))

columnas_id = ["clave_enti", "clave_muni", "clave_loca", "ageb"]

# 2203 registros de AGEBS
mask_agebs = agebs_and_external_zones[columnas_id].notna().all(axis=1)
agebs_validas = agebs_and_external_zones[mask_agebs].copy()
agebs_iniciales = (
    agebs_validas
    .drop_duplicates( # 2 vienen repetidos, se eliminan
        subset=columnas_id,
        keep="last" # el primer registro tiene geometría vacía, se conserva el último
    )
)

# 7 registros de zonas externas
zonas_externas = agebs_and_external_zones[~mask_agebs]
new_zones = gpd.GeoDataFrame(pd.concat([agebs_iniciales, zonas_externas], ignore_index=True))

# original 2203 agebs
original_zones = gpd.read_file(os.path.join(folder, "red_shapefiles", "zones_agebs_visum", "zonas_agebs_standarized.shp"))

In [65]:
print(f"Originally, there were {len(original_zones)} AGEBS in the shapefile.")
print(f"After adding the 7 external zones, there are {len(new_zones)} AGEBS in the new shapefile.")

Originally, there were 2203 AGEBS in the shapefile.
After adding the 7 external zones, there are 2210 AGEBS in the new shapefile.


In [45]:
new_zones.columns

Index(['fid', 'clave_ageb', 'clave_enti', 'clave_muni', 'clave_loca', 'ageb',
       'nombre_mun', 'tipo_ageb', 'poblacion_', 'area_m2', 'area_km2',
       'establecim', 'empleados_', 'densidad_p', 'densidad_e', 'densidad_1',
       'distancia_', 'geometry'],
      dtype='str')

In [66]:
municipio_base = {
    39: 0,       # Guadalajara
    44: 1000,    # Ixtlahuacán de los Membrillos
    51: 2000,    # Juanacatlán
    70: 3000,    # El Salto
    83: 4000,    # Tala
    97: 5000,    # Tlajomulco
    98: 6000,    # Tlaquepaque
    101: 7000,   # Tonalá
    120: 8000,   # Zapopan
    124: 9000,   # Zapotlanejo
}

accesos_carreteros = {
    "999990001": 10001, # Acceso vallarta
    "999990002": 10002, # Acceso_Colotlan
    "999990003": 10003, # Acceso_Saltillo
    "999990004": 10004, # Acceso_Zapotlanejo_Cuota
    "999990005": 10005, # Acceso_Chapala
    "999990006": 10006, # Acceso_Lopez_Mateos
    "999990007": 10007  # Acceso_zapotlanejo_Libre
}

new_zones = new_zones.rename(
    columns={
        "clave_muni": "clave_municipio",
        "nombre_mun": "nombre_municipio",
    }
).copy()

new_zones["clave_municipio"] = (
    new_zones["clave_municipio"]
    .fillna(0)
    .astype(int)
)

# Convertir clave_ageb a texto para que coincida con las llaves
new_zones["clave_ageb"] = (
    new_zones["clave_ageb"]
    .astype("string")
    .str.strip()
)

new_zones = new_zones.sort_values(
    by=["clave_municipio", "clave_ageb"]
).copy()

# Crear esta columna antes de calcular id_visum
new_zones["consecutivo_municipio"] = (
    new_zones.groupby("nombre_municipio").cumcount() + 1
)

# IDs para las zonas normales
new_zones["id_visum"] = (
    new_zones["clave_municipio"].map(municipio_base)
    + new_zones["consecutivo_municipio"]
)

# IDs para los accesos carreteros
mask_accesos = new_zones["clave_ageb"].isin(
    accesos_carreteros.keys()
)

new_zones.loc[mask_accesos, "id_visum"] = (
    new_zones.loc[mask_accesos, "clave_ageb"]
    .map(accesos_carreteros)
)

new_zones["id_visum"] = new_zones["id_visum"].astype(int)

In [69]:
new_zones.dtypes

fid                       float64
clave_ageb                 string
clave_enti                float64
clave_municipio             int64
clave_loca                float64
ageb                          str
nombre_municipio              str
tipo_ageb                     str
poblacion_                float64
area_m2                   float64
area_km2                  float64
establecim                float64
empleados_                float64
densidad_p                float64
densidad_e                float64
densidad_1                float64
distancia_                float64
geometry                 geometry
consecutivo_municipio       int64
id_visum                    int64
dtype: object

In [68]:
new_zones.to_file(f"/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/red_shapefiles/zones_agebs_visum/con accessos carreteros/new_zones_with_access.shp")

/var/folders/8s/00_wwq9j23b09mnd7gp105m00000gp/T/ipykernel_98119/4189751424.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  new_zones.to_file(f"/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/red_shapefiles/zones_agebs_visum/con accessos carreteros/new_zones_with_access.shp")
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'clave_municipio' to 'clave_muni'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'nombre_municipio' to 'nombre_mun'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'consecutivo_municipio' to 'consecutiv'
  ogr_write(
